# 🕸️ Graphs as Matrices

**A graph IS a matrix. BFS, shortest paths, and PageRank become linear algebra.**

---

## 1. Concept Intuition

An **adjacency matrix** encodes a graph: $A[i][j] = 1$ if edge from $i$ to $j$.

Key insight: $A^k[i][j]$ = number of **paths of length $k$** from $i$ to $j$.

### Why this matters
- **PageRank**: dominant eigenvector of modified adjacency matrix
- **Connectivity**: $A + A^2 + \cdots + A^n$ reveals reachability
- **BFS as matrix-vector multiply**: start vector propagated through $A$

## ⚠️ Common Wrong Intuition
*(Describe the wrong way most people first think about this. Show why it breaks, and explain how the correct intuition rectifies it.)*

## 2. Visual Representation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Social network: Alice(0), Bob(1), Carol(2), Dave(3), Eve(4)
A = np.array([[0,1,1,0,0],[0,0,1,1,0],[0,0,0,0,1],[1,0,0,0,1],[0,1,0,0,0]])
names = ['Alice','Bob','Carol','Dave','Eve']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Draw graph
angles = np.linspace(0, 2*np.pi, 5, endpoint=False)
pos = list(zip(np.cos(angles), np.sin(angles)))
for i in range(5):
    for j in range(5):
        if A[i][j] > 0:
            dx, dy = pos[j][0]-pos[i][0], pos[j][1]-pos[i][1]
            axes[0].annotate('', xy=(pos[j][0]-0.08*dx, pos[j][1]-0.08*dy),
                xytext=(pos[i][0]+0.08*dx, pos[i][1]+0.08*dy),
                arrowprops=dict(arrowstyle='->', color='#94a3b8', lw=1.5))
for i,(x,y) in enumerate(pos):
    c = plt.Circle((x,y),0.12,color='#6366f1',zorder=5)
    axes[0].add_patch(c)
    axes[0].text(x,y,names[i],ha='center',va='center',fontsize=9,fontweight='bold',color='white',zorder=6)
axes[0].set_xlim(-1.5,1.5); axes[0].set_ylim(-1.5,1.5)
axes[0].set_aspect('equal'); axes[0].axis('off')
axes[0].set_title('Social Network', fontsize=12, fontweight='bold')

# Matrix heatmap
im = axes[1].imshow(A, cmap='YlOrRd')
axes[1].set_xticks(range(5)); axes[1].set_yticks(range(5))
axes[1].set_xticklabels(names, fontsize=9, rotation=45)
axes[1].set_yticklabels(names, fontsize=9)
axes[1].set_title('Adjacency Matrix', fontsize=12, fontweight='bold')
for i in range(5):
    for j in range(5):
        axes[1].text(j,i,str(A[i,j]),ha='center',va='center',fontsize=14,
                    color='white' if A[i,j]>0 else 'black')
plt.tight_layout(); plt.show()

## 3. Mathematical Formulation

### Path counting
$A^k[i][j]$ = walks of length $k$ from $i$ to $j$

### Reachability
$(I + A + A^2 + \cdots + A^{n-1})[i][j] > 0 \iff j$ reachable from $i$

### PageRank
$\mathbf{r} = \alpha \cdot M^T \mathbf{r} + \frac{1-\alpha}{n}\mathbf{1}$, where $M$ is row-normalized $A$

## 4. Code Implementation

In [ ]:
# Path counting with matrix powers
A2, A3, A4 = A@A, A@A@A, np.linalg.matrix_power(A, 4)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (mat, t) in zip(axes, [(A,'A¹'),(A2,'A²'),(A3,'A³'),(A4,'A⁴')]):
    ax.imshow(mat, cmap='YlOrRd', vmin=0)
    ax.set_title(t, fontsize=11, fontweight='bold')
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels([n[0] for n in names], fontsize=9)
    ax.set_yticklabels([n[0] for n in names], fontsize=9)
    for i in range(5):
        for j in range(5):
            ax.text(j,i,str(int(mat[i,j])),ha='center',va='center',fontsize=11,
                   color='white' if mat[i,j]>mat.max()/2 else 'black')
plt.suptitle('Walking the Graph with Matrix Powers', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout(); plt.show()

print(f'2-step paths Alice→Eve: {int(A2[0,4])}')
print(f'3-step cycles Alice→Alice: {int(A3[0,0])}')

In [ ]:
# PageRank implementation
def pagerank(A, damping=0.85, max_iter=100):
    n = A.shape[0]
    out = A.sum(axis=1).astype(float); out[out==0] = 1
    T = A / out[:, np.newaxis]
    r = np.ones(n) / n
    for _ in range(max_iter):
        r = damping * (T.T @ r) + (1-damping)/n
    return r / r.sum()

ranks = pagerank(A.astype(float))
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#6366f1','#f97316','#10b981','#f43f5e','#8b5cf6']
bars = ax.bar(names, ranks, color=colors, alpha=0.8)
ax.set_ylabel('PageRank Score'); ax.grid(True, alpha=0.3, axis='y')
ax.set_title('Who is most important?', fontsize=13, fontweight='bold')
for b, r in zip(bars, ranks): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{r:.3f}', ha='center')
plt.tight_layout(); plt.show()

## 5. Interactive Experiment

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown

@interact(
    source=Dropdown(options=list(enumerate(names)), description='From'),
    target=Dropdown(options=list(enumerate(names)), description='To'),
    max_steps=IntSlider(value=3, min=1, max=6, description='Max steps')
)
def reachability_explorer(source, target, max_steps):
    counts = []
    Ak = np.eye(5, dtype=int)
    for k in range(1, max_steps+1):
        Ak = Ak @ A
        counts.append(int(Ak[source][target]))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(1, max_steps+1), counts,
           color=['#10b981' if c>0 else '#94a3b8' for c in counts], alpha=0.8)
    ax.set_xlabel('Path length'); ax.set_ylabel('Number of paths')
    ax.set_title(f'{names[source]} → {names[target]}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    for s, c in zip(range(1, max_steps+1), counts): ax.text(s, c+0.1, str(c), ha='center')
    plt.tight_layout(); plt.show()

## 6. Tool Exploration

In [ ]:
print('=== Degree Analysis ===')
for i, name in enumerate(names):
    print(f'  {name}: out={int(A.sum(axis=1)[i])}, in={int(A.sum(axis=0)[i])}')
print()
print('=== Eigenvalues ===')
evals = np.linalg.eigvals(A.astype(float))
print(f'Eigenvalues: {np.round(evals, 4)}')
print(f'Spectral radius: {np.max(np.abs(evals)):.4f}')
print()
print('=== Reachability Matrix ===')
reach = sum(np.linalg.matrix_power(A, k) for k in range(1, 6))
print((reach > 0).astype(int))
print(f'Strongly connected: {np.all(reach > 0)}')

## 7. Reflection Questions

1. Undirected graphs have symmetric adjacency matrices ($A = A^T$). Why? What does this imply about eigenvalues?

2. If $A^n[i][j] > 0$ for all $i,j$, the graph is strongly connected. Prove using path counting.

3. How does PageRank capture edge "quality" (not just quantity)?

4. Express BFS from node $i$ as repeated matrix-vector multiplication.

5. For weighted graphs, what does $A^k[i][j]$ mean?

---
*Graphs are matrices. → Try exercises: `02_exercises.ipynb`*

## The Feynman Technique
Explain [this concept] in plain English to someone who has never seen it. Write it in the cell below. Then check: did you use any jargon you can't define from scratch? If yes, go back.

*(Write your explanation here...)*

## Review
- **Q:** 
- **A:** 

- **Q:** 
- **A:** 

## The Takeaway
> **The one thing to carry forward is:** *(Write the insight, not the formula)*